In [1]:
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.preprocessing import normalize

In [2]:
data = pd.read_csv('glass.csv')
data.head()
n,_ = data.shape

In [16]:
data.head()

,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.0,0.0,1
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.0,0.0,1
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.0,0.0,1
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.0,0.0,1
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.0,0.0,1


In [3]:
arr = data.drop(columns=['Type']).to_numpy()

In [4]:
def minkowski_distance(x, y, p):
    distance = sum(abs(xi - yi) ** p for xi, yi in zip(x, y)) ** (1 / p)
    return distance

In [5]:
def find_maximal_cliques(adj):
    # 创建图
    G = nx.Graph(adj)
    # 找到所有极大团
    cliques = list(nx.find_cliques(G))
    return cliques

In [6]:
def closeness(i ,clique):
    if len(clique) == 1:
        return 1
    sum = 0
    for j in range(len(clique)):
        if i != j:
            sum += dis_arr[i, clique[j]]
    return (len(clique)-1)/sum

In [7]:
def sum_normalize(data):
    normalized_data = [x / sum(data) for x in data]
    return normalized_data

In [12]:
weights_ = []
for i, family in enumerate(families):
    weights = [closeness(i, clique) for clique in family]
    weights_.append(sum_normalize(weights))

In [9]:
delta = 0.25
adj_arr = np.zeros((n,n))
dis_arr = np.zeros((n,n))
for i in range(n):
    for j in range(i, n):
        dis_arr[i][j] = minkowski_distance(arr[i], arr[j], 2)
        dis_arr[j][i] = dis_arr[i][j]
        if dis_arr[i][j] < delta:
            adj_arr[i][j] = 1
            adj_arr[j][i] = 1
            
cliques = find_maximal_cliques(adj_arr)
#print(cliques)

In [10]:
families = []
for i in range(n):
    family = []
    for clique in cliques:
        if i in set(clique):
            family.append(clique);
    families.append(family)

In [13]:
classes = []
for type_, group in data.groupby('Type'):
    classes.append(group.index.to_list())

for i, family in enumerate(families):
    print(i+1, ": ")
    for j, class_ in enumerate(classes):
        pr = 0
        for k, clique in enumerate(family):
            intersection_size = len(set(clique).intersection(set(class_)))
            pr_ = intersection_size / len(clique)
            pr += pr_ * weights_[i][k]
        print('class ', j+1, pr)

1 : 
class  1 1.0
class  2 0.0
class  3 0.0
class  4 0.0
class  5 0.0
class  6 0.0
2 : 
class  1 1.0
class  2 0.0
class  3 0.0
class  4 0.0
class  5 0.0
class  6 0.0
3 : 
class  1 1.0
class  2 0.0
class  3 0.0
class  4 0.0
class  5 0.0
class  6 0.0
4 : 
class  1 0.5
class  2 0.0
class  3 0.5
class  4 0.0
class  5 0.0
class  6 0.0
5 : 
class  1 1.0
class  2 0.0
class  3 0.0
class  4 0.0
class  5 0.0
class  6 0.0
6 : 
class  1 0.5
class  2 0.5
class  3 0.0
class  4 0.0
class  5 0.0
class  6 0.0
7 : 
class  1 1.0
class  2 0.0
class  3 0.0
class  4 0.0
class  5 0.0
class  6 0.0
8 : 
class  1 1.0
class  2 0.0
class  3 0.0
class  4 0.0
class  5 0.0
class  6 0.0
9 : 
class  1 1.0
class  2 0.0
class  3 0.0
class  4 0.0
class  5 0.0
class  6 0.0
10 : 
class  1 1.0
class  2 0.0
class  3 0.0
class  4 0.0
class  5 0.0
class  6 0.0
11 : 
class  1 1.0
class  2 0.0
class  3 0.0
class  4 0.0
class  5 0.0
class  6 0.0
12 : 
class  1 1.0000000000000002
class  2 0.0
class  3 0.0
class  4 0.0
class  5 0.0

In [14]:
classes = []
for type_, group in data.groupby('Type'):
    classes.append(group.index.to_list())

# 计算每个极大团对每个类的概率
clique_probabilities = {}
for i, clique in enumerate(cliques):
    clique_probabilities[f'clique_{i+1}'] = {}
    for class_name, class_members in enumerate(classes):
        intersection_size = len(set(clique).intersection(set(class_members)))
        probability = intersection_size / len(clique)
        clique_probabilities[f'clique_{i+1}'][class_name] = probability
